# Đọc file đã được extract image feature *.b sắp sếp lại cho đúng thứ tự với df

## Bước này tạo files:

- image_feat.npy (đọc file image_features.*.b tìm có text mà không có hình fill giá trị mặc định)

In [ ]:
import os
import numpy as np
import pandas as pd

In [4]:
PATH = "./data/2023"

## Image Feature Reordering


In [ ]:
df = pd.read_parquet(os.path.join(PATH, "df_meta.2feat-encoder.parquet"))

In [ ]:
df[:5]

,itemID,asin,categories,description,title,price,imUrl,brand,related,salesRank,combined_text,sentences
0,0,097293751X,[[Baby]],Easily keep track of your baby's or child's da...,"Baby Tracker&reg; - Daily Childcare Journal, S...",17.00,http://ecx.images-amazon.com/images/I/41Bb6wf%...,Time Too,"{'also_bought': ['9729375011', 'B004FN1AE8', '...",None,"Baby Tracker&reg; - Daily Childcare Journal, S...","Baby Tracker&reg; - Daily Childcare Journal, S..."
1,1,9729375011,[[Baby]],This is version of the award-winningBaby Track...,Newborn Baby Tracker&reg; - Round the Clock Ch...,15.95,http://ecx.images-amazon.com/images/I/51r3BLpL...,,"{'also_bought': ['B000V5KPZ4', 'B001F8TLLU', '...",None,Newborn Baby Tracker&reg; - Round the Clock Ch...,Newborn Baby Tracker&reg; - Round the Clock Ch...
2,2,B00000IZQI,[[Baby]],This colorful car collection develops motor sk...,Fisher Price Nesting Action Vehicles,8.37,http://ecx.images-amazon.com/images/I/51E83QCC...,,"{'also_bought': ['B0042D69W4', 'B00428LIZM', '...",None,Fisher Price Nesting Action Vehicles This colo...,Fisher Price Nesting Action Vehicles This colo...
3,3,B00000J3LL,[[Baby]],This darling cloth book offers hands-on experi...,"My Quiet Book, Fabric Activity Book for Children",27.00,http://ecx.images-amazon.com/images/I/51GoNXhB...,,"{'also_bought': ['B00000J3LC', 'B0043G4JOA', '...",None,"My Quiet Book, Fabric Activity Book for Childr...","My Quiet Book, Fabric Activity Book for Childr..."
4,4,B00002JV9S,[[Baby]],"In a relatively new concept in teething, The F...",The First Years Massaging Action Teether,8.84,http://ecx.images-amazon.com/images/I/41gVp98n...,The First Years,"{'also_bought': ['B0013FCBJO', 'B0019QCGVK', '...",None,The First Years Massaging Action Teether The F...,The First Years Massaging Action Teether The F...


In [6]:
import array


def read_image_features(path, feature_size):
    if not os.path.exists(path):
        return
    with open(path, "rb") as f:
        while True:
            asin_bytes = f.read(10)
            if not asin_bytes:
                break
            try:
                asin = asin_bytes.decode("utf-8").strip()
                a = array.array("f")
                # Đọc đúng số lượng float bạn yêu cầu
                a.fromfile(f, feature_size)
                yield asin, a.tolist()
            except EOFError:
                break
            except Exception as e:
                print(f"Lỗi tại vị trí {f.tell()}: {e}")
                break

In [7]:
# --- CẤU HÌNH ---
FEATURE_SIZE = 4096
FILE_B_PATH = os.path.join(PATH, "image_feature.vgg16.b")

In [8]:
image_features = read_image_features(FILE_B_PATH, feature_size=FEATURE_SIZE)

In [44]:
# Lấy thử 1 phần tử đầu tiên
first_item = next(image_features)

# Kiểm tra hình dạng
print(f"Kiểu dữ liệu của 1 phần tử: {type(first_item)}")
print(f"Mã ASIN (ID sản phẩm): {first_item[0]}")
print(f"Độ dài vector ảnh: {len(first_item[1])}")
print(f"5 giá trị đầu tiên trong vector: {first_item[1][:5]}")

Kiểu dữ liệu của 1 phần tử: <class 'tuple'>
Mã ASIN (ID sản phẩm): B004LE8ZYO
Độ dài vector ảnh: 4096
5 giá trị đầu tiên trong vector: [0.0, 0.0, 0.0, 0.0, 0.0]


In [45]:
first_item[1][:10]

[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.6348890066146851,
 0.0,
 0.10182002186775208,
 0.0,
 0.0]

In [ ]:
from tqdm import tqdm

num_items = len(df)
OUTPUT_NPY = os.path.join(PATH, "image_feat.npy")

# 1. Map ASIN -> itemID
map_asin_itemID = dict(zip(df["asin"], df["itemID"]))

# 2. Khởi tạo ma trận và biến hỗ trợ tính trung bình
final_matrix = np.zeros((num_items, FEATURE_SIZE), dtype=np.float32)
filled_indices = set()
running_sum = np.zeros(
    FEATURE_SIZE, dtype=np.float64
)  # Dùng float64 để tránh tràn số khi cộng dồn

print(f"🚀 Đang trích xuất ảnh cho {num_items} sản phẩm...")

# 3. VỪA ĐỌC VỪA ĐIỀN + TÍNH TỔNG CỘNG DỒN
for asin, feat_list in tqdm(
    read_image_features(FILE_B_PATH, FEATURE_SIZE), desc="Processing"
):
    if asin in map_asin_itemID:
        target_idx = int(map_asin_itemID[asin])
        feat_array = np.array(feat_list, dtype=np.float32)

        final_matrix[target_idx] = feat_array
        filled_indices.add(target_idx)

        # Cộng dồn để lát nữa tính trung bình (chỉ tính trên những cái có dữ liệu thật)
        running_sum += feat_array

# 4. XỬ LÝ DỮ LIỆU THIẾU (Lấp đầy bằng Average Vector)
all_indices = set(range(num_items))
missing_indices = sorted(list(all_indices - filled_indices))

if len(filled_indices) > 0:
    # Tính vector trung bình từ những item CÓ ảnh
    avg_vector = (running_sum / len(filled_indices)).astype(np.float32)

    if missing_indices:
        print(
            f"⚠️ Cảnh báo: Thiếu {len(missing_indices)} ảnh. Đang điền bằng vector trung bình..."
        )
        # Điền vector trung bình vào các vị trí trống bằng vectorization (nhanh hơn loop)
        final_matrix[missing_indices] = avg_vector

        # Lưu log danh sách ID thiếu để bạn kiểm tra đồ án
        err_log = os.path.join(PATH, "missed_img_itemIDs.csv")
        np.savetxt(err_log, missing_indices, delimiter=",", fmt="%d")
else:
    print("❌ LỖI NGHIÊM TRỌNG: Không tìm thấy bất kỳ ảnh nào khớp với dữ liệu!")
    # Tùy chọn: điền toàn bộ bằng 0 hoặc raise lỗi nếu cần

# 5. LƯU KẾT QUẢ
np.save(OUTPUT_NPY, final_matrix)

print(f"✅ Hoàn thành! File đã lưu: {OUTPUT_NPY}")
print(f"Kích thước ma trận: {final_matrix.shape}")
print(f"Số lượng item dùng ảnh mặc định: {len(missing_indices)}")

🚀 Đang trích xuất ảnh cho 7050 sản phẩm...


Processing: 7037it [00:00, 7160.80it/s]


⚠️ Cảnh báo: Thiếu 13 ảnh. Đang điền bằng vector trung bình...
✅ Hoàn thành! File đã lưu: ./data/2014\image_feat.npy
Kích thước ma trận: (7050, 4096)
Số lượng item dùng ảnh mặc định: 13
